# EvolAI — Fine-tune on Colab

**Workflow:**
1. Run on your VPS first: `evolcli miner get-challenge 55 --netuid 47 -o challenge.json`
2. Upload `challenge.json` to this notebook (Cell 3)
3. Run all cells → get a commit hash at the end
4. Back on VPS: `evolcli miner register ... --revision <commit-hash>`

**Runtime:** Runtime → Change runtime type → **T4 GPU** (free)

## Cell 1 — Install dependencies

In [ ]:
!pip install -q transformers==4.47.0 trl==0.12.0 peft==0.14.0 datasets huggingface_hub accelerate bitsandbytes

## Cell 2 — Your credentials (fill these in)

In [ ]:
from google.colab import userdata

# Add your token in Colab: left sidebar → 🔑 Secrets → Add new secret
# Name: HF_TOKEN   Value: your HuggingFace token (hf_...)
HF_TOKEN   = userdata.get("HF_TOKEN")
MODEL_REPO = "Roystar/evolai-qwen2.5-1.5b"

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to HuggingFace")

## Cell 3 — Upload challenge.json

**On your VPS, run:**
```bash
evolcli miner get-challenge 55 --netuid 47 -o challenge.json
```
Then upload `challenge.json` using the file picker below.

In [ ]:
from google.colab import files
import json

uploaded = files.upload()  # select challenge.json from your computer

filename = list(uploaded.keys())[0]
challenge = json.loads(uploaded[filename])

epoch = challenge.get('epoch', '?')
n_val = challenge.get('validator_count', 0)
union = challenge.get('union', {})
total_rows = sum(v['count'] for v in union.values())

print(f"Epoch       : {epoch}")
print(f"Validators  : {n_val}")
print(f"Total rows  : {total_rows}")
for ds, info in union.items():
    print(f"  {ds}: {info['count']} indices → {info['indices'][:5]}...")

## Cell 4 — Load training data from HuggingFace datasets

In [ ]:
from datasets import load_dataset, Dataset

_INSTR_COLS = ("instruction", "input", "question", "prompt", "human")
_RESP_COLS  = ("response", "output", "answer", "completion", "assistant", "gpt")

def extract_sample(row):
    keys = {k.lower(): k for k in row.keys()}
    instr_key = next((keys[c] for c in _INSTR_COLS if c in keys), None)
    resp_key  = next((keys[c] for c in _RESP_COLS  if c in keys), None)
    if instr_key and resp_key:
        return {"instruction": str(row[instr_key]).strip(),
                "response":    str(row[resp_key]).strip()}
    for v in row.values():
        if isinstance(v, str) and v.strip():
            return {"instruction": v.strip(), "response": ""}
    return None

samples = []
for ds_name, ds_info in union.items():
    indices = ds_info["indices"]
    print(f"Loading {len(indices)} rows from {ds_name} ...")
    ds = load_dataset(ds_name, split="train")
    for row in ds.select(indices):
        s = extract_sample(dict(row))
        if s:
            samples.append(s)

print(f"\nTotal samples: {len(samples)}")
print("\nExample:")
print("  instruction:", samples[0]['instruction'][:100])
print("  response   :", samples[0]['response'][:100])

## Cell 5 — Load model + tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False
print(f"\nModel loaded: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")

## Cell 6 — Attach LoRA adapters

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 7 — Format dataset + train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

def format_sample(s):
    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "user",      "content": s["instruction"]},
            {"role": "assistant", "content": s["response"]},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False)
    return f"### Human: {s['instruction']}\n\n### Assistant: {s['response']}"

train_dataset = Dataset.from_list([{"text": format_sample(s)} for s in samples])
print(f"Training on {len(train_dataset)} samples")

training_args = TrainingArguments(
    output_dir="/tmp/model_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=2048,
)

trainer.train()
print("Training complete!")

## Cell 8 — Merge LoRA into base model

In [ ]:
print("Merging LoRA weights ...")
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained("/tmp/model_output", safe_serialization=True)
tokenizer.save_pretrained("/tmp/model_output")
print("Saved to /tmp/model_output/")

## Cell 9 — Push to HuggingFace + get commit hash

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone

tag = datetime.now(timezone.utc).strftime("v%Y%m%d-%H%M%S")
api = HfApi(token=HF_TOKEN)

api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True)

print(f"Uploading to {MODEL_REPO} ...")
commit_info = api.upload_folder(
    folder_path="/tmp/model_output",
    repo_id=MODEL_REPO,
    repo_type="model",
    commit_message=f"EvolAI auto-train epoch={epoch} {tag}",
)

COMMIT_HASH = getattr(commit_info, "oid", None) or tag

print()
print("=" * 60)
print("UPLOAD COMPLETE")
print("=" * 60)
print(f"Commit hash: {COMMIT_HASH}")
print()
print("Now run this on your VPS:")
print(f"""evolcli miner register {MODEL_REPO} \\
  --wallet-name powallet \\
  --hotkey pohotkey \\
  --track transformer \\
  --revision {COMMIT_HASH}""")

## Done!

Copy the `evolcli miner register ...` command printed above and run it on your VPS.

**Repeat every epoch (~72 minutes)** to keep showing improvement and climb the rankings.